In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("multipleDatasetsApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/25 21:14:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
yellowTaxiSchema = StructType([
 StructField("VendorID", IntegerType(), True),
 StructField("tpep_pickup_datetime", TimestampType(), True),
 StructField("tpep_dropoff_datetime", TimestampType(), True),
 StructField("passenger_count", DoubleType(), True),
 StructField("trip_distance", DoubleType(), True),
 StructField("RatecodeID", DoubleType(), True),
 StructField("store_and_fwd_flag", StringType(), True),
 StructField("PULocationID", IntegerType(), True),
 StructField("DOLocationID", IntegerType(), True),
 StructField("payment_type", IntegerType(), True),
 StructField("fare_amount", DoubleType(), True),
 StructField("extra", DoubleType(), True),
 StructField("mta_tax", DoubleType(), True),
 StructField("tip_amount", DoubleType(), True),
 StructField("tolls_amount", DoubleType(), True),
 StructField("improvement_surcharge", DoubleType(), True),
 StructField("total_amount", DoubleType(), True),
 StructField("congestion_surcharge", DoubleType(), True),
 StructField("airport_fee", DoubleType(), True),
])

In [5]:
yellowTaxisDF = spark.read.option("header", "true").schema(yellowTaxiSchema).csv(
    "./Files/YellowTaxis_202210.csv"
)
yellowTaxisDF.createOrReplaceTempView("YellowTaxis")
yellowTaxisDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



25/05/25 21:14:27 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
taxiZonesSchema = 'PickupLocationID INT, Borough STRING, Zone STRING, ServiceZone STRING'

taxiZonesDF = spark.read.schema(taxiZonesSchema).csv(
    "./Files/TaxiZones.csv"
)

taxiZonesDF.createOrReplaceGlobalTempView("TaxiZones")

taxiZonesDF.printSchema()

root
 |-- PickupLocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- ServiceZone: string (nullable = true)



In [8]:
joinedDF = yellowTaxisDF.join(
    taxiZonesDF,
    yellowTaxisDF.PULocationID == taxiZonesDF.PickupLocationID,
    "inner"
).drop(
    col("PickupLocationID"))
joinedDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- ServiceZone: string (nullable = true)



In [9]:
driversDF = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "./Files/Drivers.csv"
)
driversDF.createOrReplaceTempView("Drivers")
driversDF.show()

+-------------------+--------------------+--------------------+--------------+---------------+
|DriverLicenseNumber|                Name|                Type|ExpirationDate|LastDateUpdated|
+-------------------+--------------------+--------------------+--------------+---------------+
|            5430898|   ABDEL-BAR,ESLAM,M|MEDALLION TAXI DR...|    04/12/2023|     04/22/2020|
|            5363749|ABDOUSAMADOV,ALIC...|MEDALLION TAXI DR...|    06/01/2020|     04/22/2020|
|            5534446|  ABDUHALIKOV,RUSTAM|MEDALLION TAXI DR...|    06/16/2020|     04/22/2020|
|            5935702|   ABDULLAEV,JONIBEK|MEDALLION TAXI DR...|    03/14/2022|     04/22/2020|
|            5255097|ABDULNABI,MASHHOUR,H|MEDALLION TAXI DR...|    03/16/2021|     04/22/2020|
|            5778633|ABDUSALOMOV,IKROMJON|MEDALLION TAXI DR...|    06/02/2023|     04/22/2020|
|            5934755|ABDUVOKHIDOV,MURO...|MEDALLION TAXI DR...|    02/27/2022|     04/22/2020|
|             443085|         ABEDIN,MD,J|MEDALLIO

In [10]:
cabsDF = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "./Files/Cabs.csv"
)
cabsDF.createOrReplaceTempView("Cabs")
cabsDF.show()

+---------+--------------------+--------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+--------------------+--------------------+---------------+
|CabNumber|VehicleLicenseNumber|                Name|     LicenseType|Active|PermitLicenseNumber| VehicleVinNumber|WheelchairAccessible|VehicleYear|VehicleType|TelephoneNumber|             Website|             Address|LastDateUpdated|
+---------+--------------------+--------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+--------------------+--------------------+---------------+
| T802127C|              C19641|          ABCON INC.|OWNER MUST DRIVE|   YES|               NULL|5TDBK3EH0DS268018|                NULL|       2016|       NULL|  (718)438-1100|                NULL|41-24   38 STREET...|     04/22/2020|
| T525963C|             5362996| ACCEPTABLE TAXI LLC|    NAM

In [11]:
spark.sql("(SELECT Name FROM Cabs WHERE LicenseType = 'OWNER MUST DRIVE') UNION ALL (SELECT Name FROM Drivers)").count()

157716

In [12]:
spark.sql("(SELECT Name FROM Cabs WHERE LicenseType = 'OWNER MUST DRIVE') UNION (SELECT Name FROM Drivers)").count()

156566

In [13]:
spark.sql("(SELECT Name FROM Cabs WHERE LicenseType = 'OWNER MUST DRIVE') INTERSECT (SELECT Name FROM Drivers)").count()

1150

In [14]:
spark.sql("(SELECT Name FROM Cabs WHERE LicenseType = 'OWNER MUST DRIVE') EXCEPT (SELECT Name FROM Drivers)").count()

1940